# Qwen3.5 0.8B — LoRA SFT for Malaysia Moderation

Fine-tune `Qwen/Qwen3.5-0.8B` with LoRA adapters on the Malaysia moderation dataset.
Structured JSON output: the model judges content safety and generates rewrites.

**Hardware:** Kaggle T4 GPU (16GB VRAM)
**Dataset:** 6,047 train / 726 validation examples (augmented with edge cases)
**Method:** LoRA (r=16, alpha=32) with SFTTrainer

In [ ]:
!pip install -q transformers>=4.49 peft>=0.13 trl>=0.12 datasets>=3.0 accelerate>=1.0 bitsandbytes>=0.44

In [ ]:
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

import shared_utils as utils

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
TRAIN_PATH = "train.jsonl"
VAL_PATH = "validation.jsonl"

train_raw = utils.load_jsonl(TRAIN_PATH)
val_raw = utils.load_jsonl(VAL_PATH)

print(f"Train: {len(train_raw)} examples")
print(f"Validation: {len(val_raw)} examples")

train_texts = [utils.format_qwen(ex) for ex in train_raw]
val_texts = [utils.format_qwen(ex) for ex in val_raw]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

print("\n--- Example formatted text ---")
print(train_texts[0][:500])

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules="all-linear",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
OUTPUT_DIR = "qwen35-malaysia-mod"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_seq_length=1024,
    dataset_text_field="text",
    report_to="none",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print(f"Adapter saved to {OUTPUT_DIR}/final_adapter")

for log in trainer.state.log_history:
    if "eval_loss" in log:
        print(f"Epoch {log.get('epoch', '?')}: eval_loss={log['eval_loss']:.4f}")

## Quick Sanity Check
Run inference on a few examples to verify the model produces valid JSON.

In [ ]:
model.eval()

test_inputs = [
    "Hari ini cuaca sangat panas. Jom minum air kelapa kat pantai!",
    "Semua orang kaum X memang pemalas dan suka curi.",
    "Government should consider raising minimum wage to match inflation rates.",
    "'Balik tongsan lah kalau tak suka!' — overheard this at the kopitiam. 2024 and people still talking like this. So sad.",
    "Orang tua aku selalu pesan: hormati semua agama, semua kaum. Kita sama-sama anak Malaysia.",
]

for text in test_inputs:
    prompt = utils.inference_prompt_qwen(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            temperature=1.0,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    parsed = utils.parse_model_output(response)

    print(f"\nInput: {text[:80]}...")
    print(f"Valid JSON: {parsed is not None}")
    if parsed:
        print(f"  safe={parsed.get('safe')}, severity={parsed.get('severity')}")
        print(f"  reason={parsed.get('reason', '')[:80]}")
    print()